# Idea on Spectral Granger Causality

In [ ]:
from mne_connectivity import spectral_connectivity_epochs
import numpy as np

data = filtered.to_numpy().T[np.newaxis, :, :]  # (1 x 13 x 2000)
n_neurons = data.shape[1]
neuron_names_unsorted = filtered.columns

gc_matrix = np.zeros((n_neurons, n_neurons))
trgc_matrix = np.zeros((n_neurons, n_neurons))

for i in range(n_neurons):
    for j in range(n_neurons):
        if i != j:
            indices_ij = (np.array([[i]]), np.array([[j]]))
            indices_ji = (np.array([[j]]), np.array([[i]]))
            
            # forward GC
            gc_ij = spectral_connectivity_epochs(
                data, method=['gc'],
                indices=indices_ij,
                sfreq=20,
                fmin=0.05, fmax=1,
                gc_n_lags=40,
                mt_bandwidth=0.5,  # multitaper bandwidth
                verbose=False
            )
            gc_ji = spectral_connectivity_epochs(
                data, method=['gc'],
                indices=indices_ji,
                sfreq=20,
                fmin=0.05, fmax=1,
                gc_n_lags=40,
                mt_bandwidth=0.5,
                verbose=False
            )
            
            # time-reversed GC for significance
            gc_tr_ij = spectral_connectivity_epochs(
                data, method=['gc_tr'],
                indices=indices_ij,
                sfreq=20,
                fmin=0.05, fmax=1,
                gc_n_lags=40,
                mt_bandwidth=0.5,
                verbose=False
            )
            gc_tr_ji = spectral_connectivity_epochs(
                data, method=['gc_tr'],
                indices=indices_ji,
                sfreq=20,
                fmin=0.05, fmax=1,
                gc_n_lags=40,
                mt_bandwidth=0.5,
                verbose=False
            )
            
            # net GC
            net_gc = np.mean(gc_ij.get_data() - gc_ji.get_data())
            gc_matrix[i, j] = net_gc
            
            # TRGC: net GC - net time-reversed GC
            # positive TRGC means i truly drives j (not spurious)
            net_gc_tr = np.mean(gc_tr_ij.get_data() - gc_tr_ji.get_data())
            trgc_matrix[i, j] = net_gc - net_gc_tr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(gc_matrix, cmap='RdBu_r')
plt.colorbar(im1, ax=axes[0], label='Net GC (i->j)')
axes[0].set_xticks(range(n_neurons))
axes[0].set_xticklabels(neuron_names_unsorted, rotation=45)
axes[0].set_yticks(range(n_neurons))
axes[0].set_yticklabels(neuron_names_unsorted)
axes[0].set_title('Net Granger Causality')

im2 = axes[1].imshow(trgc_matrix, cmap='RdBu_r')
plt.colorbar(im2, ax=axes[1], label='TRGC (i->j)')
axes[1].set_xticks(range(n_neurons))
axes[1].set_xticklabels(neuron_names_unsorted, rotation=45)
axes[1].set_yticks(range(n_neurons))
axes[1].set_yticklabels(neuron_names_unsorted)
axes[1].set_title('Time-Reversed GC (spurious-corrected)')

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [ ]:

n_neurons = data.shape[1]

sources = []
targets = []

for i in range(n_neurons):
    for j in range(n_neurons):
        if i != j:
            sources.append([i])   # <-- wrap in list
            targets.append([j])   # <-- wrap in list

indices = (np.array(sources), np.array(targets))

gc = spectral_connectivity_epochs(
    data,
    method='gc',
    indices=indices,
    sfreq=20,
    fmin=0.05,
    fmax=1,
    gc_n_lags=80,
    mt_bandwidth=0.05,
    verbose=False
)

gc_tr = spectral_connectivity_epochs(
    data,
    method='gc_tr',   
    indices=indices,
    sfreq=20,
    fmin=0.05,
    fmax=1,
    gc_n_lags=80,
    mt_bandwidth=0.05,
    verbose=False
)

In [ ]:
gc_data = gc.get_data()  # (n_connections, n_freqs)
freqs = gc.freqs

n_freqs = len(freqs)
gc_matrix_full = np.zeros((n_neurons, n_neurons, n_freqs))

for idx, (src, tgt) in enumerate(zip(indices[0], indices[1])):
    i = src[0]
    j = tgt[0]
    gc_matrix_full[i, j, :] = gc_data[idx]


gc_tr_data = gc_tr.get_data()
freqs = gc_tr.freqs

gc_tr_matrix_full = np.zeros((n_neurons, n_neurons, n_freqs))

for idx,(src,tgt) in enumerate(zip(indices[0], indices[1])):
    i = src[0]
    j = tgt[0]
    gc_tr_matrix_full[i,j,:] = gc_tr_data[idx]

In [ ]:

gc_matrix_full.shape
# (n_neurons, n_neurons, n_freqs)
gc_values = gc_matrix_full.copy()

# remove diagonal
for i in range(n_neurons):
    gc_values[i, i, :] = np.nan

ymin = np.nanmin(gc_values)
ymax = np.nanmax(gc_values)


In [ ]:

fig, axes = plt.subplots(
    n_neurons,
    n_neurons,
    figsize=(12, 12),
    sharex=True,
    sharey=True
)

for i in range(n_neurons):
    for j in range(n_neurons):
        ax = axes[i, j]
        
        if i == j:
            ax.axis("off")
        else:
            ax.plot(freqs, gc_matrix_full[i, j, :])
            ax.set_ylim(ymin, ymax)

        if i == n_neurons - 1:
            ax.set_xlabel("Freq")
        if j == 0:
            ax.set_ylabel("GC")

plt.suptitle("Multivariate Spectral GC (i → j)")
plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(
    n_neurons,
    n_neurons,
    figsize=(12, 12),
    sharex=True,
    sharey=True
)

for i in range(n_neurons):
    for j in range(n_neurons):
        ax = axes[i, j]
        
        if i == j:
            ax.axis("off")
        else:
            ax.plot(freqs, gc_tr_matrix_full[i, j, :])
            ax.set_ylim(ymin, ymax)

        if i == n_neurons - 1:
            ax.set_xlabel("Freq")
        if j == 0:
            ax.set_ylabel("GC")

plt.suptitle("Multivariate Spectral GC (i → j)")
plt.tight_layout()
plt.show()

In [ ]:
net_gc_full = gc_matrix_full - np.transpose(gc_matrix_full, (1, 0, 2))



n_neurons = net_gc_full.shape[0]

# Compute global y limits (exclude diagonal)
net_vals = net_gc_full.copy()
for i in range(n_neurons):
    net_vals[i, i, :] = np.nan

ymin = np.nanmin(net_vals)
ymax = np.nanmax(net_vals)

fig, axes = plt.subplots(
    n_neurons,
    n_neurons,
    figsize=(14, 14),
    sharex=True,
    sharey=True
)

for i in range(n_neurons):
    for j in range(n_neurons):
        ax = axes[i, j]

        if i >= j:
            ax.axis("off")
        else:
            ax.plot(freqs, net_gc_full[i, j, :])
            ax.axhline(0, color="k", linestyle=":", linewidth=0.5)
            ax.set_ylim(ymin, ymax)
            ax.set_title(
                f"{neuron_names_unsorted[i]} ↔ {neuron_names_unsorted[j]}",
                fontsize=7
            )
plt.suptitle("Net Multivariate Spectral Granger Causality", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
net_gc_tr_full = gc_tr_matrix_full - np.transpose(gc_tr_matrix_full, (1, 0, 2))


In [ ]:
trgc_full = net_gc_full - net_gc_tr_full

In [ ]:
# Compute global y limits (exclude diagonal)
trgc_vals = trgc_full.copy()
for i in range(n_neurons):
    trgc_vals[i, i, :] = np.nan

ymin = np.nanmin(trgc_vals)
ymax = np.nanmax(trgc_vals)

fig, axes = plt.subplots(
    n_neurons,
    n_neurons,
    figsize=(14, 14),
    sharex=True,
    sharey=True
)

for i in range(n_neurons):
    for j in range(n_neurons):
        ax = axes[i, j]

        ax.plot(freqs, trgc_full[i, j, :])
        ax.axhline(0, color="k", linestyle=":", linewidth=0.5)
        #ax.set_ylim(ymin, ymax)
        ax.set_title(
            f"{neuron_names_unsorted[i]} ↔ {neuron_names_unsorted[j]}",
            fontsize=7
        )
plt.suptitle("Net Multivariate Spectral Granger Causality", fontsize=16)
plt.tight_layout()
plt.show()

# Ideas on ranking the population by phase diff

In [ ]:
complex_diff = np.exp(1j * (phase_diff_sorted.to_numpy()))

masked_complex = np.where(p_val_matrix.to_numpy() < 0.05, complex_diff, np.nan)
np.fill_diagonal(masked_complex, np.nan)

mean_offset = np.angle(np.nanmean(masked_complex, axis=1))
rank_order = np.argsort(mean_offset)

In [ ]:
n_significant_pairs = np.sum(
    (p_val_matrix.to_numpy() < 0.05) & ~np.eye(len(p_val_matrix), dtype=bool), 
    axis=1
)
neuron_names = phase_diff_sorted.columns.tolist()

for rank, i in enumerate(rank_order):
    print(f"Rank {rank+1} - {neuron_names[i]}: {np.degrees(mean_offset[i]):.3f} degrees, {n_significant_pairs[i]} significant pairs")

Rank 1 -  C06: -59.465 degrees, 7 significant pairs
Rank 2 -  C07: -32.965 degrees, 7 significant pairs
Rank 3 -  C12: -30.640 degrees, 7 significant pairs
Rank 4 -  C00: -0.307 degrees, 8 significant pairs
Rank 5 -  C01: 7.544 degrees, 4 significant pairs
Rank 6 -  C05: 17.035 degrees, 8 significant pairs
Rank 7 -  C04: 21.762 degrees, 8 significant pairs
Rank 8 -  C02: 29.852 degrees, 8 significant pairs
Rank 9 -  C03: 33.816 degrees, 7 significant pairs


In [ ]:
from scipy import stats
import pingouin as pg


# collect per-neuron phase offsets as a list of arrays
neuron_offsets = []
for i in range(len(neuron_names)):
    # get significant pairs for neuron i
    sig_mask = (p_val_matrix.to_numpy()[i] < 0.05)
    sig_mask[i] = False  # exclude self
    offsets = phase_diff_sorted.to_numpy()[i, sig_mask]
    neuron_offsets.append(offsets)

# Kruskal-Wallis test (non-parametric, works on circular data if differences are small)
stat, p = stats.kruskal(*[o for o in neuron_offsets if len(o) > 0])
print(f"Kruskal-Wallis: H={stat:.3f}, p={p:.4f}")

Kruskal-Wallis: H=33.271, p=0.0001


In [ ]:
# build long format
rows = []
for i, offsets in enumerate(neuron_offsets):
    if len(offsets) > 0:
        for val in offsets:
            rows.append({'neuron': neuron_names[i], 'offset': val})

df_offsets = pd.DataFrame(rows)

# pairwise Mann-Whitney with Bonferroni correction
result = pg.pairwise_tests(data=df_offsets, dv='offset', between='neuron', 
                            parametric=False, padjust='bonf')
print(result[['A', 'B', 'U_val', 'p_unc', 'p_corr']].to_string())

       A     B  U_val     p_unc    p_corr
0    C00   C01   12.0  0.569697  1.000000
1    C00   C02   15.0  0.082984  1.000000
2    C00   C03   12.0  0.072106  1.000000
3    C00   C04   17.0  0.130381  1.000000
4    C00   C05   17.0  0.130381  1.000000
5    C00   C06   52.0  0.003730  0.134266
6    C00   C07   46.0  0.040093  1.000000
7    C00   C12   46.0  0.040093  1.000000
8    C01   C02   10.0  0.367677  1.000000
9    C01   C03    8.0  0.315152  1.000000
10   C01   C04   14.0  0.808081  1.000000
11   C01   C05   17.0  0.933333  1.000000
12   C01   C06   28.0  0.006061  0.218182
13   C01   C07   24.0  0.072727  1.000000
14   C01   C12   23.0  0.109091  1.000000
15   C02   C03   26.0  0.866511  1.000000
16   C02   C04   41.0  0.382284  1.000000
17   C02   C05   43.0  0.278632  1.000000
18   C02   C06   56.0  0.000311  0.011189
19   C02   C07   52.0  0.003730  0.134266
20   C02   C12   51.0  0.005905  0.212587
21   C03   C04   38.0  0.280963  1.000000
22   C03   C05   39.0  0.231857  1

In [ ]:
# observed statistic: variance of circular means across neurons
observed_means = np.array([pg.circ_mean(o) for o in neuron_offsets if len(o) > 0])
observed_stat = np.var(observed_means)

# permutation test: shuffle neuron labels
all_offsets = np.concatenate([o for o in neuron_offsets if len(o) > 0])
group_sizes = [len(o) for o in neuron_offsets if len(o) > 0]

n_perm = 1000
perm_stats = []
for _ in range(n_perm):
    shuffled = np.random.permutation(all_offsets)
    perm_means = []
    idx = 0
    for size in group_sizes:
        perm_means.append(pg.circ_mean(shuffled[idx:idx+size]))
        idx += size
    perm_stats.append(np.var(perm_means))

p_value = np.mean(np.array(perm_stats) >= observed_stat)
print(f"Permutation test p-value: {p_value:.4f}")
print(f"Observed variance of circular means: {observed_stat:.4f}")

Permutation test p-value: 0.0000
Observed variance of circular means: 0.2842


In [ ]:
from itertools import combinations

# observed pairwise differences between circular means
observed_means = np.array([pg.circ_mean(o) for o in neuron_offsets if len(o) > 0])
valid_neurons = [neuron_names[i] for i, o in enumerate(neuron_offsets) if len(o) > 0]

all_offsets = np.concatenate([o for o in neuron_offsets if len(o) > 0])
group_sizes = [len(o) for o in neuron_offsets if len(o) > 0]

n_perm = 1000
pairwise_results = {}

for (idx_a, idx_b) in combinations(range(len(observed_means)), 2):
    observed_diff = np.abs(np.angle(np.exp(1j * (observed_means[idx_a] - observed_means[idx_b]))))
    
    perm_diffs = []
    for _ in range(n_perm):
        shuffled = np.random.permutation(all_offsets)
        perm_means = []
        idx = 0
        for size in group_sizes:
            perm_means.append(pg.circ_mean(shuffled[idx:idx+size]))
            idx += size
        perm_diffs.append(np.abs(np.angle(np.exp(1j * (perm_means[idx_a] - perm_means[idx_b])))))
    
    p = np.mean(np.array(perm_diffs) >= observed_diff)
    pairwise_results[(valid_neurons[idx_a], valid_neurons[idx_b])] = p

# print with Bonferroni correction
n_comparisons = len(pairwise_results)
for (a, b), p in pairwise_results.items():
    p_corr = min(p * n_comparisons, 1.0)
    print(f"{a} vs {b}: p={p:.4f}, p_bonf={p_corr:.4f}")

 C01 vs  C12: p=0.1670, p_bonf=1.0000
 C01 vs  C03: p=0.3300, p_bonf=1.0000
 C01 vs  C00: p=0.7760, p_bonf=1.0000
 C01 vs  C02: p=0.4100, p_bonf=1.0000
 C01 vs  C04: p=0.6190, p_bonf=1.0000
 C01 vs  C05: p=0.7370, p_bonf=1.0000
 C01 vs  C06: p=0.0120, p_bonf=0.4320
 C01 vs  C07: p=0.1260, p_bonf=1.0000
 C12 vs  C03: p=0.0040, p_bonf=0.1440
 C12 vs  C00: p=0.1720, p_bonf=1.0000
 C12 vs  C02: p=0.0060, p_bonf=0.2160
 C12 vs  C04: p=0.0240, p_bonf=0.8640
 C12 vs  C05: p=0.0230, p_bonf=0.8280
 C12 vs  C06: p=0.2320, p_bonf=1.0000
 C12 vs  C07: p=0.9140, p_bonf=1.0000
 C03 vs  C00: p=0.1270, p_bonf=1.0000
 C03 vs  C02: p=0.8630, p_bonf=1.0000
 C03 vs  C04: p=0.5890, p_bonf=1.0000
 C03 vs  C05: p=0.4610, p_bonf=1.0000
 C03 vs  C06: p=0.0000, p_bonf=0.0000
 C03 vs  C07: p=0.0010, p_bonf=0.0360
 C00 vs  C02: p=0.1680, p_bonf=1.0000
 C00 vs  C04: p=0.3220, p_bonf=1.0000
 C00 vs  C05: p=0.4240, p_bonf=1.0000
 C00 vs  C06: p=0.0090, p_bonf=0.3240
 C00 vs  C07: p=0.1560, p_bonf=1.0000
 C02 vs  C04

# Population graph idea

In [ ]:
import networkx as nx

G = nx.DiGraph()
for i in range(len(neuron_names)):
    for j in range(len(neuron_names)):
        if i != j and p_val_matrix.to_numpy()[i,j] < 0.05:
            if phase_diff_sorted.to_numpy()[i,j] > 0:
                G.add_edge(neuron_names[i], neuron_names[j], 
                          weight=PLV_sorted.to_numpy()[i,j])

# check in/out degree
for node in G.nodes():
    print(f"{node}: out={G.out_degree(node)}, in={G.in_degree(node)}, "
          f"weighted_out={G.out_degree(node, weight='weight'):.3f}, "
          f"weighted_in={G.in_degree(node, weight='weight'):.3f}")

# check for cycles
cycles = list(nx.simple_cycles(G))
print(f"\nNumber of cycles: {len(cycles)}")
print(cycles)

 C01: out=4, in=0, weighted_out=1.026, weighted_in=0.000
 C00: out=3, in=5, weighted_out=2.082, weighted_in=3.879
 C02: out=7, in=1, weighted_out=5.818, weighted_in=0.216
 C04: out=5, in=3, weighted_out=4.015, weighted_in=2.091
 C05: out=4, in=4, weighted_out=3.066, weighted_in=3.049
 C12: out=2, in=5, weighted_out=1.342, weighted_in=3.817
 C06: out=0, in=7, weighted_out=0.000, weighted_in=4.738
 C07: out=1, in=6, weighted_out=0.655, weighted_in=3.945
 C03: out=6, in=1, weighted_out=4.606, weighted_in=0.875

Number of cycles: 0
[]


In [ ]:
n_perm = 1000
random_cycles = []

n_edges = G.number_of_edges()
nodes = list(G.nodes())

for _ in range(n_perm):
    G_rand = nx.DiGraph()
    G_rand.add_nodes_from(nodes)
    
    # randomly assign same number of edges among nodes
    possible_edges = [(i, j) for i in nodes for j in nodes if i != j]
    random_edges = np.random.choice(len(possible_edges), n_edges, replace=False)
    
    for idx in random_edges:
        i, j = possible_edges[idx]
        G_rand.add_edge(i, j)
    
    cycles_rand = list(nx.simple_cycles(G_rand))
    random_cycles.append(len(cycles_rand))

print(f"Observed cycles: {len(cycles)}")
print(f"Random mean cycles: {np.mean(random_cycles):.2f} ± {np.std(random_cycles):.2f}")
print(f"p-value: {np.mean((np.array(random_cycles)+1) <= (len(cycles))+1):.4f}")

Observed cycles: 0
Random mean cycles: 262.72 ± 86.00
p-value: 0.0000


In [ ]:
topo_order = list(nx.topological_sort(G))
print("Topological order (leader to lagger):")
for rank, node in enumerate(topo_order):
    print(f"Rank {rank+1} - {node}: out={G.out_degree(node)}, in={G.in_degree(node)}, "
          f"weighted_out={G.out_degree(node, weight='weight'):.3f}")

Topological order (leader to lagger):
Rank 1 -  C01: out=4, in=0, weighted_out=1.026
Rank 2 -  C02: out=7, in=1, weighted_out=5.818
Rank 3 -  C03: out=6, in=1, weighted_out=4.606
Rank 4 -  C04: out=5, in=3, weighted_out=4.015
Rank 5 -  C05: out=4, in=4, weighted_out=3.066
Rank 6 -  C00: out=3, in=5, weighted_out=2.082
Rank 7 -  C12: out=2, in=5, weighted_out=1.342
Rank 8 -  C07: out=1, in=6, weighted_out=0.655
Rank 9 -  C06: out=0, in=7, weighted_out=0.000


In [ ]:
pos = nx.spring_layout(G, seed=42)
weights = [G[u][v]['weight'] for u,v in G.edges()]

nx.draw(G, pos, with_labels=True, 
        node_color='lightblue',
        edge_color=weights, edge_cmap=plt.cm.Reds,
        width=[w*3 for w in weights],
        arrows=True, arrowsize=20)
plt.title('Phase leadership graph (edge weight = PLV)')
plt.show()

In [ ]:
G2 = nx.DiGraph()
for i in range(len(neuron_names)):
    for j in range(len(neuron_names)):
        if i != j and p_val_matrix.to_numpy()[i,j] < 0.05:
            diff = phase_diff_sorted.to_numpy()[i,j]
            if diff > 0:
                # i leads j by diff radians
                G2.add_edge(neuron_names[i], neuron_names[j],
                           weight=np.degrees(diff))  # in degrees for readability

for node in G2.nodes():
    in_edges = G2.in_edges(node, data='weight')
    out_edges = G2.out_edges(node, data='weight')
    
    in_weights = [w for _,_,w in in_edges]
    out_weights = [w for _,_,w in out_edges]
    
    mean_in = np.mean(in_weights) if in_weights else 0
    mean_out = np.mean(out_weights) if out_weights else 0
    
    print(f"{node}: out={G2.out_degree(node)}, in={G2.in_degree(node)}, "
          f"mean_lead={mean_out:.2f}°, mean_lag={mean_in:.2f}°")

cycles = list(nx.simple_cycles(G2))
print(f"\nNumber of cycles: {len(cycles)}")

if len(cycles) == 0:
    topo_order2 = list(nx.topological_sort(G2))
    print("\nTopological order:")
    for rank, node in enumerate(topo_order2):
        print(f"Rank {rank+1} - {node}")

In [ ]:
# how many pairs have no edge in either direction?
n_unconstrained = 0
for i, node_i in enumerate(neuron_names):
    for j, node_j in enumerate(neuron_names):
        if i < j:
            if not G2.has_edge(node_i, node_j) and not G2.has_edge(node_j, node_i):
                n_unconstrained += 1
                print(f"Unconstrained: {node_i} - {node_j}")

print(f"\nTotal unconstrained pairs: {n_unconstrained}")
print(f"Total possible pairs: {len(neuron_names)*(len(neuron_names)-1)//2}")

NameError: name 'G2' is not defined

# Ideas on the comparison with the population

In [ ]:
phase_matrix = filtered_phase.to_numpy()  # already extracted phases
phase_pop = np.angle(np.mean(np.exp(1j * phase_matrix), axis=1)) #circular mean of individual neurons

population_diff = np.angle(np.exp(1j * (filtered_phase.to_numpy() - phase_pop[:, np.newaxis]))) 


pop_mean_offset = np.angle(np.mean(np.exp(1j * population_diff), axis=0))
pop_rank_order = np.argsort(pop_mean_offset)

neuron_names = filtered_phase.columns.tolist()
sorted_names = [neuron_names[i] for i in pop_rank_order]
sorted_pop_diff = population_diff[:, pop_rank_order]

fig, ax = plt.subplots(figsize=(12, 5))
ax.violinplot(sorted_pop_diff, showmedians=True, showextrema=False)
ax.axhline(0, color='red')
ax.set_xticks(range(1, len(sorted_names) + 1))
ax.set_xticklabels(sorted_names, rotation=45)
ax.set_ylabel('Phase offset from population (rad)')
ax.set_title('Neurons sorted by phase offset from population')
plt.tight_layout()
plt.show()